# 00 — Seed a lousy baseline model (pre-demo)

Run **before** the presenter flow (notebooks 01–05).

Goal: put a deliberately weak ONNX on S3 at `object-detection/xray-detector/baseline` and stand up the InferenceService on that path so the live demo already has a **prev** revision. Notebooks 01–04 then train/register a better model; canary rolls a real second pod.

This notebook is intentionally under-trained (`fraction` ≪ 1, 1 epoch, small `imgsz`) so the later canary looks better — not for accuracy.

Requires: GPU preferred, `AWS_*` / Connection, `oc` logged in (for apply).

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "images" / "data.yaml").exists() and (REPO_ROOT.parent / "images" / "data.yaml").exists():
    REPO_ROOT = REPO_ROOT.parent
    os.chdir(REPO_ROOT)
sys.path.insert(0, str(REPO_ROOT))

import mlflow
from ultralytics import YOLO, settings

settings.update({"mlflow": False})

from src.mlflow_utils import configure_tracking, log_dataset_lineage, log_yolo_metrics
from src.s3_utils import baseline_storage_path, upload_file

DATA_YAML = "images/data.yaml"
EXPERIMENT = configure_tracking()
print("cwd:", Path.cwd())
print("MLflow tracking URI:", mlflow.get_tracking_uri())
print("Experiment:", EXPERIMENT)
print("Baseline storage path:", baseline_storage_path())

## Train a weak detector

Knobs (env overrides):

| Env | Default | Effect |
|-----|---------|--------|
| `BASELINE_EPOCHS` | `1` | Minimal fit |
| `BASELINE_FRACTION` | `0.02` | Use ~2% of train images |
| `BASELINE_IMGSZ` | `320` | Small input |
| `DEMO_BATCH` / `DEMO_WORKERS` | `8` / `2` | Same 16Gi-friendly defaults as 01 |
| `DEMO_MODEL` | `yolov8n.pt` | Starting weights |

In [ ]:
EPOCHS = int(os.environ.get("BASELINE_EPOCHS", "1"))
FRACTION = float(os.environ.get("BASELINE_FRACTION", "0.02"))
IMGSZ = int(os.environ.get("BASELINE_IMGSZ", "320"))
BASE_MODEL = os.environ.get("DEMO_MODEL", "yolov8n.pt")
BATCH = int(os.environ.get("DEMO_BATCH", "8"))
WORKERS = int(os.environ.get("DEMO_WORKERS", "2"))
RUN_NAME = os.environ.get("BASELINE_RUN_NAME", "baseline-lousy")

print(
    f"epochs={EPOCHS} fraction={FRACTION} imgsz={IMGSZ} "
    f"batch={BATCH} workers={WORKERS} model={BASE_MODEL}"
)

with mlflow.start_run(run_name=RUN_NAME) as run:
    run_id = run.info.run_id
    mlflow.log_params({
        "model": BASE_MODEL,
        "data": DATA_YAML,
        "epochs": EPOCHS,
        "imgsz": IMGSZ,
        "fraction": FRACTION,
        "batch": BATCH,
        "workers": WORKERS,
        "run_name": RUN_NAME,
        "project": "runs-openshift",
        "role": "demo-baseline-seed",
    })
    log_dataset_lineage(DATA_YAML)

    model = YOLO(BASE_MODEL)
    results = model.train(
        data=DATA_YAML,
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        workers=WORKERS,
        fraction=FRACTION,
        cache=False,
        amp=True,
        plots=False,
        project="runs-openshift",
        name=RUN_NAME,
        exist_ok=True,
    )

    recorded = {}
    for metrics in [
        getattr(results, "results_dict", None),
        results if isinstance(results, dict) else None,
        getattr(model, "metrics", None),
    ]:
        if metrics is None:
            continue
        recorded = log_yolo_metrics(metrics)
        if recorded.get("mAP50") is not None:
            break
    if recorded.get("mAP50") is None:
        try:
            recorded = log_yolo_metrics(model.val(data=DATA_YAML, fraction=min(1.0, FRACTION * 5)))
        except Exception as exc:  # noqa: BLE001
            print(f"warning: val metrics skipped: {exc}")

    weights_dir = Path("runs") / "detect" / "runs-openshift" / RUN_NAME / "weights"
    if not (weights_dir / "best.pt").is_file():
        alt = Path("runs-openshift") / RUN_NAME / "weights"
        if (alt / "best.pt").is_file():
            weights_dir = alt
    best_pt = weights_dir / "best.pt"
    assert best_pt.is_file(), f"Missing weights: {best_pt}"
    mlflow.log_artifact(str(best_pt), artifact_path="weights")
    mlflow.set_tag("weights_local_path", str(best_pt))
    mlflow.set_tag("role", "demo-baseline-seed")

print(f"Finished {RUN_NAME} run_id={run_id} metrics={recorded}")
print("best.pt:", best_pt)

## Export ONNX → upload as baseline → apply InferenceService

Uploads to `{BASELINE_PATH}/1/model.onnx` (OVMS layout), then runs `openshift/ensure-baseline-isvc.sh --force` so the ISVC serves that path.

In [ ]:
import subprocess

export_model = YOLO(str(best_pt))
onnx_path = Path(export_model.export(format="onnx", imgsz=IMGSZ))
if not onnx_path.is_file():
    onnx_path = best_pt.with_name("best.onnx")
print("ONNX:", onnx_path)

baseline = baseline_storage_path()
key = f"{baseline}/1/model.onnx"
s3_uri = upload_file(onnx_path, key)
# Flat sibling helps older tooling / head checks
upload_file(onnx_path, f"{baseline}/model.onnx")
print("Uploaded baseline:", s3_uri)

with mlflow.start_run(run_id=run_id):
    mlflow.log_param("baseline_s3_uri", s3_uri)
    mlflow.set_tag("baseline_storage_path", baseline)
    if onnx_path.is_file():
        mlflow.log_artifact(str(onnx_path), artifact_path="onnx")

script = REPO_ROOT / "openshift" / "ensure-baseline-isvc.sh"
env = os.environ.copy()
env["SOURCE_PATH"] = baseline
env["BASELINE_PATH"] = baseline
print("Running:", script, "--force")
proc = subprocess.run(
    ["bash", str(script), "--force"],
    cwd=str(REPO_ROOT),
    env=env,
    capture_output=True,
    text=True,
)
print(proc.stdout)
if proc.returncode != 0:
    print(proc.stderr)
    raise SystemExit(
        f"ensure-baseline-isvc.sh failed ({proc.returncode}). "
        "Baseline is in S3; apply manually: ./openshift/ensure-baseline-isvc.sh --force"
    )
print("Baseline InferenceService should be Ready on", baseline)
print("Next: notebooks/01_train_track.ipynb")